In [ ]:
import torch
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision import models

In [ ]:
weights = ViT_B_16_Weights.DEFAULT
model = vit_b_16(weights=weights).eval()



In [ ]:
model = models.resnet18()
model.fc = torch.nn.Linear(model.fc.in_features, 4)  # 1 for confidence + 3 for vector
state_dict = torch.load("train/best_model.pth")['model_state_dict']
model.load_state_dict(state_dict, strict=True)

False


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [10]:
from pyk4a import PyK4A
import numpy as np
import cv2
import einops
from pyk4a import CalibrationType
from pyk4a import Config, ColorResolution, DepthMode, FPS
def draw_vector(img_path, model):
    base_name = img_path[:-4]
    label_path = base_name + ".txt"
    labels = np.loadtxt(label_path)
    wrist_coords = labels[1:4]
    cfg = Config(
        color_resolution=ColorResolution.RES_1080P,       # 1920x1080
        depth_mode=DepthMode.NFOV_UNBINNED,               # 640x576 depth
        synchronized_images_only=True,                     # depth+color in same capture
        camera_fps= FPS.FPS_15
    )
    k4a = PyK4A(cfg)
    k4a.start()
    calib = k4a.calibration
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    rgb_img = img_rgb.astype(np.float32) / 255.0  # Normalize to [0, 1]
    rgb_img = einops.rearrange(rgb_img, 'h w c -> c h w')  # (3, H, W)
    input_tensor = torch.tensor(rgb_img).unsqueeze(0)  # (1, 3, H, W)
    with torch.no_grad():
        output = model(input_tensor)
    vector = output[0, 1:].cpu().numpy()
    vector = vector / np.linalg.norm(vector)  # Normalize the vector
    xmm, ymm, zmm = wrist_coords * 1000  # Convert to mm
    start_point = np.array([xmm, ymm, zmm])
    end_point = start_point + vector * 300  # Scale the vector for visualization

    uv = calib.convert_3d_to_2d(end_point, CalibrationType.COLOR, CalibrationType.COLOR)
    camera_coords_calculated = tuple(map(int, uv))

    uv = calib.convert_3d_to_2d((xmm, ymm, zmm), CalibrationType.COLOR, CalibrationType.COLOR)
    camera_coords_wrist = tuple(map(int, uv))
    cv2.line(img, camera_coords_wrist, camera_coords_calculated, (0, 255, 0), 2)
    cv2.imshow("Image with Pointing Vector", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

draw_vector("data\\11-11-170343.jpg", model)